In [1]:
import json
import sqlite3
from typing import List, Dict, Any
from openai import OpenAI
import os
import requests as r

# Fetch data from the OData API
url = 'https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter=EmployeeID%20eq%203%20and%20ShippedDate%20le%20RequiredDate&$format=json'
response = r.get(url)
res1 = response.json()
json_data = res1['value']

class JSONalyze:
    def __init__(self):
        self.conn = sqlite3.connect(':memory:')
        self.cursor = self.conn.cursor()
        self.table_name = 'json_data'
        self.columns = []

        # Set up OpenAI client
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        if not self.client:
            raise ValueError("OpenAI API key not set in environment variables.")

    def load_json(self, json_data: List[Dict[str, Any]]):
        if not json_data:
            raise ValueError("JSON data is empty")

        self.columns = list(json_data[0].keys())

        # Create table with TEXT type for all columns
        columns = ', '.join([f'"{k}" TEXT' for k in self.columns])
        self.cursor.execute(f'CREATE TABLE {self.table_name} ({columns})')

        # Insert data
        placeholders = ', '.join(['?' for _ in self.columns])
        for item in json_data:
            values = [item.get(k) for k in self.columns]
            self.cursor.execute(f'INSERT INTO {self.table_name} VALUES ({placeholders})', values)

        self.conn.commit()

    def inspect_column_types(self):
        self.cursor.execute(f'PRAGMA table_info({self.table_name})')
        return {row[1]: row[2] for row in self.cursor.fetchall()}  # {column_name: data_type}

    def natural_language_to_sql(self, query: str) -> str:
        prompt = f"""
        Convert the following natural language query to SQL. 
        The table name is '{self.table_name}' and it has the following columns: {', '.join(self.columns)}.
        Provide only the SQL query without any additional formatting or explanation.
        
        Natural language query: {query}
        
        SQL query:
        """

        response = self.client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that converts natural language queries to SQL. Provide only the SQL query without any additional formatting or explanation."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=150,
            n=1,
            stop=None,
            temperature=0.1,
        )

        if response.choices:
            sql_query = response.choices[0].message.content.strip()
            sql_query = sql_query.replace('```sql', '').replace('```', '').strip()
            return sql_query
        else:
            raise ValueError("No SQL query returned from OpenAI API.")

    def adjust_query_for_types(self, sql_query: str) -> str:
        column_types = self.inspect_column_types()

        for column, data_type in column_types.items():
            if data_type == 'TEXT' and column in sql_query:
                if any(op in sql_query for op in ['>', '<', '>=', '<=', '=']):
                    if 'Freight' in column or 'Cost' in column:  # Adjust for numeric fields
                        sql_query = sql_query.replace(f"{column} >", f"CAST({column} AS REAL) >")
                        sql_query = sql_query.replace(f"{column} <", f"CAST({column} AS REAL) <")
                        sql_query = sql_query.replace(f"{column} >=", f"CAST({column} AS REAL) >=")
                        sql_query = sql_query.replace(f"{column} <=", f"CAST({column} AS REAL) <=")
                        sql_query = sql_query.replace(f"{column} =", f"CAST({column} AS REAL) =")
                    elif 'Date' in column:  # Adjust for date fields
                        sql_query = sql_query.replace(f"{column} =", f"DATE({column}) =")
                        sql_query = sql_query.replace(f"{column} >", f"DATE({column}) >")
                        sql_query = sql_query.replace(f"{column} <", f"DATE({column}) <")
                        sql_query = sql_query.replace(f"{column} >=", f"DATE({column}) >=")
                        sql_query = sql_query.replace(f"{column} <=", f"DATE({column}) <=")

        return sql_query

    def execute_sql(self, sql_query: str) -> str:
        sql_query_adjusted = self.adjust_query_for_types(sql_query)
        print(f"Executing SQL query: {sql_query_adjusted}")  # For debugging

        try:
            self.cursor.execute(sql_query_adjusted)
            columns = [description[0] for description in self.cursor.description]
            results = [dict(zip(columns, row)) for row in self.cursor.fetchall()]

            # Format the output
            if results:
                output = '\n'.join(', '.join(f"{key}: {value}" for key, value in row.items()) for row in results)
                return f"Results:\n{output}"
            else:
                return "No results found."
        except sqlite3.Error as e:
            return f"An error occurred: {e}"

    def print_table(self):
        try:
            self.cursor.execute(f'SELECT * FROM {self.table_name}')
            columns = [description[0] for description in self.cursor.description]
            rows = self.cursor.fetchall()

            # Print column names
            print(f"Table: {self.table_name}")
            print(f"{' | '.join(columns)}")
            print('-' * (len(columns) * 20))

            # Print rows
            for row in rows:
                print(' | '.join(str(cell) for cell in row))
        except sqlite3.Error as e:
            print(f"An error occurred: {e}")

    def close(self):
        self.conn.close()

# Example usage
if __name__ == "__main__":
    jsonalyze = JSONalyze()
    jsonalyze.load_json(json_data)

    # Print the entire table
    # jsonalyze.print_table()

    # # Execute a direct SQL query and print the result
    # sql_query = "SELECT * FROM json_data WHERE ShipCountry = 'Germany' AND CAST(Freight AS REAL) > 600;"
    # print(jsonalyze.execute_sql(sql_query))

    # Alternatively, execute a query using natural language
    natural_query = "How many orders have been handled by employeeID 3? How many of those have been shipped on time"
    sql_query_from_nl = jsonalyze.natural_language_to_sql(natural_query)
    print(jsonalyze.execute_sql(sql_query_from_nl))

    jsonalyze.close()


Executing SQL query: SELECT COUNT(OrderID)
FROM json_data
WHERE EmployeeID = 3
AND DATE(ShippedDate) <= RequiredDate;
Results:
COUNT(OrderID): 122
